In [ ]:
PipelineRunId = "MANUAL_RUN"

In [1]:
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath
import hashlib, os, re, shutil, zipfile
import pandas as pd

Main_Root = Path('/lakehouse/default/Files/UL_Router') 
ROOT = Path('/lakehouse/default/Files/UL_Router/GDF')
INCOMING_ZIPS = ROOT / 'IncomingZip'
MAPPING_FILE = Main_Root / 'Configuration' / 'Mapping.xlsx'
READY_OUTPUT = ROOT / 'ReadyForSharePoint1'
AUDIT_FOLDER = ROOT / 'Audit'

COUNTRY_SHEET = 'Country'
CATEGORY_SHEET = 'Category'
FILE_TYPE_SHEET = 'File Type'
BRAND_PREFIX = 'UB'
COUNTRY_CODE_LENGTH = 2
CLEAR_READY_OUTPUT_AT_START = True
SKIP_EXISTING = True
MAX_FILES_PER_ZIP = 1000
MAX_UNCOMPRESSED_BYTES = 20 * 1024**3
DEFAULT_FILE_TYPE_MAP = {
    'METADATA': 'Metadata',
    'CASEDATA': 'Case Data',
    'TABLES': 'Tables',
    'EXISTENCE': 'Existence',
    'CHECKFILE': "CheckFile",
}

for p in [INCOMING_ZIPS, MAPPING_FILE.parent, READY_OUTPUT, AUDIT_FOLDER]:
    p.mkdir(parents=True, exist_ok=True)
if CLEAR_READY_OUTPUT_AT_START and READY_OUTPUT.exists():
    shutil.rmtree(READY_OUTPUT)
    READY_OUTPUT.mkdir(parents=True, exist_ok=True)

print('ZIP staging input:', INCOMING_ZIPS)
print('Mapping staging input:', MAPPING_FILE)
print('Final hierarchy staging output:', READY_OUTPUT)

StatementMeta(, da9f6dc6-14d7-40ef-9602-0d1617851763, 3, Finished, Available, Finished, False)

ZIP staging input: /lakehouse/default/Files/UL_Router/IncomingZip
Mapping staging input: /lakehouse/default/Files/UL_Router/Configuration/Mapping.xlsx
Final hierarchy staging output: /lakehouse/default/Files/UL_Router/ReadyForSharePoint1


In [2]:
if not MAPPING_FILE.exists():
    raise FileNotFoundError(f'Missing {MAPPING_FILE}. Run Copy_Mapping_To_Lakehouse first.')

xls = pd.ExcelFile(MAPPING_FILE, engine='openpyxl')
country_df = pd.read_excel(xls, sheet_name=COUNTRY_SHEET, dtype=str)
category_df = pd.read_excel(xls, sheet_name=CATEGORY_SHEET, dtype=str)
country_df.columns = country_df.columns.astype(str).str.strip()
category_df.columns = category_df.columns.astype(str).str.strip()

def find_col(columns, names):
    names = {n.casefold() for n in names}
    return next((x for x in columns if x.strip().casefold() in names), None)

def make_lookup(df, key_col, value_col, label):
    t = df[[key_col, value_col]].dropna().copy()
    t[key_col] = t[key_col].astype(str).str.strip().str.upper()
    t[value_col] = t[value_col].astype(str).str.strip()
    t = t[(t[key_col] != '') & (t[value_col] != '')].drop_duplicates()
    conflicts = t.groupby(key_col)[value_col].nunique()
    conflicts = conflicts[conflicts > 1]
    if not conflicts.empty:
        raise ValueError(f'Conflicting {label} mappings: {conflicts.index.tolist()}')
    return dict(zip(t[key_col], t[value_col]))

country_code_col = find_col(country_df.columns, ['Cont', 'Country Code', 'CountryCode'])
country_name_col = find_col(country_df.columns, ['Country'])
category_code_col = find_col(category_df.columns, ['CATEGORY'])
category_name_col = find_col(category_df.columns, ['CATEGORY NAME'])
dashboard_col = find_col(category_df.columns, ['Dashboard Category'])
if dashboard_col is None:
    dashboard_col = find_col(category_df.columns, ['SUB CATEGORY'])

if not all([country_code_col, country_name_col, category_code_col, category_name_col, dashboard_col]):
    raise ValueError('Required mapping columns are missing.')

country_map = make_lookup(country_df, country_code_col, country_name_col, 'country')
dashboard_map = make_lookup(category_df, category_code_col, dashboard_col, 'dashboard category')
category_name_map = make_lookup(category_df, category_code_col, category_name_col, 'category name')

if FILE_TYPE_SHEET in xls.sheet_names:
    file_type_df = pd.read_excel(xls, sheet_name=FILE_TYPE_SHEET, dtype=str)
    file_type_df.columns = file_type_df.columns.astype(str).str.strip()
    if file_type_df.shape[1] < 2:
        raise ValueError('File Type sheet needs a code column and folder column.')
    file_type_map = make_lookup(file_type_df, file_type_df.columns[0], file_type_df.columns[1], 'file type')
else:
    file_type_map = DEFAULT_FILE_TYPE_MAP.copy()

print(f'Loaded {len(country_map)} countries, {len(dashboard_map)} categories and {len(file_type_map)} file types.')
print('Dashboard folder source column:', dashboard_col)

StatementMeta(, da9f6dc6-14d7-40ef-9602-0d1617851763, 4, Finished, Available, Finished, False)

Loaded 29 countries, 57 categories and 5 file types.
Dashboard folder source column: Dashboard Category


In [3]:
def safe_folder(value):
    value = re.sub(r'[\r\n]+', ' - ', str(value).strip())
    value = re.sub(r'\s+', ' ', value)
    value = re.sub(r'[~"#%*:<>?/\\{|}]', '-', value).rstrip('. ')
    if not value:
        raise ValueError('Mapped folder name is blank.')
    return value


def parse_name(file_name):
    name = Path(file_name).name
    parts = Path(name).stem.upper().split('_')
    if len(parts) < 3:
        raise ValueError(f'Unexpected filename pattern: {name}')
    identity, file_type_code = parts[0], parts[1]
    if not identity.startswith(BRAND_PREFIX):
        raise ValueError(f'Unexpected brand prefix: {name}')

    tail = identity[len(BRAND_PREFIX):]
    country_code = tail[:COUNTRY_CODE_LENGTH]
    category_code = tail[COUNTRY_CODE_LENGTH:]
    if country_code not in country_map:
        raise ValueError(f'Unknown country code {country_code}: {name}')
    if category_code not in dashboard_map:
        raise ValueError(f'Unknown category code {category_code}: {name}')
    if file_type_code not in file_type_map:
        raise ValueError(f'Unknown file type {file_type_code}: {name}')

    country = safe_folder(country_map[country_code])
    dashboard = safe_folder(dashboard_map[category_code])
    file_type_folder = safe_folder(file_type_map[file_type_code])
    relative = Path(country) / dashboard / file_type_folder / name
    return {
        'file_name': name,
        'country_code': country_code,
        'country_name': country_map[country_code],
        'category_code': category_code,
        'category_name': category_name_map.get(category_code),
        'dashboard_category': dashboard_map[category_code],
        'file_type_code': file_type_code,
        'file_type_folder': file_type_folder,
        'relative_output_path': relative.as_posix(),
        'final_sharepoint_path': f'Unilever Global PBI/Processed Data/{relative.as_posix()}',
    }

sample = parse_name('UBJPHSOF_METADATA_202301_202630_20260908.CSV')
display(pd.DataFrame([sample]))
assert sample['final_sharepoint_path'].startswith('Unilever Global PBI/Processed Data/Japan/Hair Care/Metadata/')

StatementMeta(, da9f6dc6-14d7-40ef-9602-0d1617851763, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1f3ad66c-46f0-4050-80a5-884957af2034)

In [4]:
def validate_zip(archive):
    members = [m for m in archive.infolist() if not m.is_dir()]
    if len(members) > MAX_FILES_PER_ZIP:
        raise ValueError('ZIP file count exceeds the configured safety limit.')
    if sum(m.file_size for m in members) > MAX_UNCOMPRESSED_BYTES:
        raise ValueError('ZIP expanded size exceeds the configured safety limit.')
    for member in members:
        path = PurePosixPath(member.filename.replace('\\', '/'))
        if path.is_absolute() or '..' in path.parts:
            raise ValueError(f'Unsafe ZIP member: {member.filename}')
    return members


def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as source:
        for chunk in iter(lambda: source.read(8 * 1024**2), b''):
            digest.update(chunk)
    return digest.hexdigest()


def extract_member(archive, member, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and SKIP_EXISTING:
        return 'SKIPPED_ALREADY_EXISTS'
    partial = destination.with_suffix(destination.suffix + '.partial')
    with archive.open(member) as source, open(partial, 'wb') as target:
        shutil.copyfileobj(source, target, length=8 * 1024**2)
    if partial.stat().st_size != member.file_size:
        partial.unlink(missing_ok=True)
        raise RuntimeError('Extracted file size mismatch.')
    os.replace(partial, destination)
    return 'READY_FOR_SHAREPOINT'

zip_paths = sorted(INCOMING_ZIPS.glob('*.zip'))
if not zip_paths:
    raise FileNotFoundError(
        f'No ZIP files found in {INCOMING_ZIPS}. '
        'The pipeline must first copy direct ZIP files from SharePoint Documents/Raw data.'
    )

records = []
summary_rows = []
for zip_path in zip_paths:
    zip_ok = True
    extracted_count = 0
    try:
        if not zipfile.is_zipfile(zip_path):
            raise ValueError('Invalid ZIP archive.')
        with zipfile.ZipFile(zip_path) as archive:
            for member in validate_zip(archive):
                file_name = Path(member.filename).name
                row = {'run_utc': datetime.now(timezone.utc).isoformat(), 'zip_name': zip_path.name, 'file_name': file_name, 'size_bytes': member.file_size}
                try:
                    route = parse_name(file_name)
                    row.update(route)
                    destination = READY_OUTPUT / route['relative_output_path']
                    status = extract_member(archive, member, destination)
                    row.update({'status': status, 'sha256': sha256(destination), 'message': 'Success'})
                    extracted_count += 1
                    print(status, route['final_sharepoint_path'])
                except Exception as exc:
                    zip_ok = False
                    row.update({'status': 'FAILED', 'message': str(exc)})
                    print('FAILED', file_name, exc)
                records.append(row)
        zip_status = 'SUCCESS' if zip_ok else 'FAILED'
    except Exception as exc:
        zip_status = 'ZIP_LEVEL_FAILURE'
        records.append({'run_utc': datetime.now(timezone.utc).isoformat(), 'zip_name': zip_path.name, 'status': zip_status, 'message': str(exc)})
    summary_rows.append({'zip_name': zip_path.name, 'extracted_file_count': extracted_count, 'status': zip_status})

audit_df = pd.DataFrame(records)
summary_df = pd.DataFrame(summary_rows)
display(summary_df)
display(audit_df)

StatementMeta(, da9f6dc6-14d7-40ef-9602-0d1617851763, 6, Finished, Available, Finished, False)

READY_FOR_SHAREPOINT Unilever Global PBI/Processed Data/Brazil/Hair Care/Case Data/UBBRHSOF_CASEDATA_202301_202630_20260905.CSV
READY_FOR_SHAREPOINT Unilever Global PBI/Processed Data/Brazil/Hair Care/Existence/UBBRHSOF_EXISTENCE_202301_202630_20260905.CSV
READY_FOR_SHAREPOINT Unilever Global PBI/Processed Data/Brazil/Hair Care/Tables/UBBRHSOF_TABLES_202301_202630_20260905.xlsx
READY_FOR_SHAREPOINT Unilever Global PBI/Processed Data/Brazil/Hair Care/Metadata/UBBRHSOF_METADATA_202301_202630_20260908.CSV
READY_FOR_SHAREPOINT Unilever Global PBI/Processed Data/Brazil/Hair Care/CheckFile/UBBRHSOF_CheckFile_202301_202634.xlsx
READY_FOR_SHAREPOINT Unilever Global PBI/Processed Data/Brazil/Hair Care/Case Data/UBBRHSOF_CASEDATA_202627_202634_20260914.CSV
READY_FOR_SHAREPOINT Unilever Global PBI/Processed Data/Brazil/Hair Care/Existence/UBBRHSOF_EXISTENCE_202627_202634_20260914.CSV
READY_FOR_SHAREPOINT Unilever Global PBI/Processed Data/Brazil/Hair Care/Metadata/UBBRHSOF_METADATA_202627_202634_

SynapseWidget(Synapse.DataFrame, e6d58b7f-2f7b-4104-8c4e-17e7ceb41f95)

SynapseWidget(Synapse.DataFrame, 7e544b74-96df-4ea9-9448-93705da01860)

In [5]:
stamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
audit_path = AUDIT_FOLDER / f'file_audit_{stamp}.csv'
summary_path = AUDIT_FOLDER / f'zip_summary_{stamp}.csv'
audit_df.to_csv(audit_path, index=False, encoding='utf-8-sig')
summary_df.to_csv(summary_path, index=False, encoding='utf-8-sig')
failed = audit_df[audit_df['status'].isin(['FAILED', 'ZIP_LEVEL_FAILURE'])] if not audit_df.empty else pd.DataFrame()
print('Final hierarchy ready at:', READY_OUTPUT)
print('Audit:', audit_path)
if not failed.empty:
    raise RuntimeError(f'{len(failed)} failed record(s). Review {audit_path}.')
print({'status': 'SUCCESS', 'file_count': len(audit_df), 'ready_output': str(READY_OUTPUT)})

StatementMeta(, da9f6dc6-14d7-40ef-9602-0d1617851763, 7, Finished, Available, Finished, False)

Final hierarchy ready at: /lakehouse/default/Files/UL_Router/ReadyForSharePoint1
Audit: /lakehouse/default/Files/UL_Router/Audit/file_audit_20260922_172122.csv


RuntimeError: 3 failed record(s). Review /lakehouse/default/Files/UL_Router/Audit/file_audit_20260922_172122.csv.